# BusinessGPT Model Evaluation

One Save & Run job generates one frozen HF profile. Private prompts and outputs stay in Kaggle input/output; this notebook contains no upload code.

In [ ]:
%%capture
%pip uninstall -y torchao
%pip install git+https://github.com/huggingface/transformers.git
%pip install --no-cache-dir --upgrade peft accelerate huggingface_hub


In [ ]:
from importlib.util import find_spec

if find_spec("torchao") is not None:
    raise RuntimeError("Incompatible optional torchao package is still installed; PEFT cannot load adapters safely.")

import accelerate
import huggingface_hub
import peft
import torch
import transformers

print("Dependency versions:")
for package in (torch, transformers, peft, accelerate, huggingface_hub):
    print(f"  {package.__name__}={package.__version__}")


In [ ]:
# Change only these experiment parameters between Save & Run jobs.
PROFILE_ID = "v16_hf_production"
DATASET_PATH = "/kaggle/input/datasets/avxofi/businessgpt-eval/temporal_eval.json"
EXPECTED_DATASET_ID = "businessgpt_temporal_ead8a3ff2196"
OUTPUT_PATH = f"/kaggle/working/{PROFILE_ID}.jsonl"
REPO_URL = "https://github.com/vXofi/businessgpt.git"
REPO_REF = "main"


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret("HF_TOKEN"))


In [ ]:
import subprocess
import sys
from pathlib import Path

repo = Path("/kaggle/working/businessgpt")
if not repo.is_dir():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(repo)])
revision = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(repo))
print(f"Evaluation code revision: {revision}")

from eval.model_eval.common import resolve_json_dataset_path

DATASET_PATH = str(resolve_json_dataset_path(
    DATASET_PATH,
    search_root="/kaggle/input",
    expected_dataset_id=EXPECTED_DATASET_ID,
))
print(f"Resolved private dataset: {DATASET_PATH}")


In [ ]:
import json

from eval.model_eval.generation import generate_hf, validate_generation_output

manifest_path = repo / "eval/experiments/v16_baseline.json"
stats = generate_hf(
    manifest_path=manifest_path,
    dataset_path=DATASET_PATH,
    profile_id=PROFILE_ID,
    output_path=OUTPUT_PATH,
)
validation = validate_generation_output(
    manifest_path=manifest_path,
    dataset_path=DATASET_PATH,
    profile_id=PROFILE_ID,
    output_path=OUTPUT_PATH,
)
print("Generation stats:", json.dumps(stats, indent=2))
print("Artifact validation:", json.dumps(validation, indent=2))
print(f"Kaggle output: {OUTPUT_PATH}")
